In [1]:
# ==========================
# PHASE 3 — FEATURE ENGINEERING
# ==========================

import pandas as pd
import numpy as np
import re
import os

from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import StandardScaler

In [2]:
#cell 2

train = pd.read_csv("/Users/amarachi/Documents/AQA/Exp_3/phase2/Notebook/strict_splits/train.csv")
val = pd.read_csv("/Users/amarachi/Documents/AQA/Exp_3/phase2/Notebook/strict_splits/val.csv")
test = pd.read_csv("/Users/amarachi/Documents/AQA/Exp_3/phase2/Notebook/strict_splits/test.csv")

print("Train:",len(train))
print("Validation:",len(val))
print("Test:",len(test))

Train: 2338
Validation: 621
Test: 355


In [3]:
#cell 3
assert "text" in train.columns
assert "label" in train.columns

print("Dataset structure verified.")

Dataset structure verified.


In [4]:
#cell 4
vague_terms = [
    "adequate","appropriate","sufficient","efficient",
    "timely","quick","fast","user-friendly",
    "as needed","where applicable","if necessary"
]

weak_modals = [
    "may","should","could","can","might"
]

quantifiers = [
    "some","many","few","several","various"
]

frequency_terms = [
    "often","sometimes","periodically","occasionally"
]

In [5]:
# CELL 5 — Improved Rule-Based Feature Extractor

def rule_features(text):

    text = text.lower()

    tokens = text.split()

    # Existing features
    weak_modal = int(any(w in tokens for w in weak_modals))
    vague = int(any(v in text for v in vague_terms))
    quantifier = int(any(q in tokens for q in quantifiers))
    frequency = int(any(f in tokens for f in frequency_terms))

    clause_density = text.count(",") + len(re.findall(r"\band\b", text)) + len(re.findall(r"\bor\b", text))

    # New stronger signals
    passive = int(bool(re.search(r"\b(is|was|were|be|been|being)\b\s+\w+ed\b", text)))

    conditional = int(bool(re.search(r"\b(if|unless|until|when|where)\b", text)))

    sentence_length = len(tokens)

    return [
        weak_modal,
        vague,
        quantifier,
        frequency,
        clause_density,
        passive,
        conditional,
        sentence_length
    ]

In [6]:
#cell 6
train_rules = np.array(train["text"].apply(rule_features).tolist())
val_rules = np.array(val["text"].apply(rule_features).tolist())
test_rules = np.array(test["text"].apply(rule_features).tolist())

print("Rule feature shape:",train_rules.shape)

Rule feature shape: (2338, 8)


In [7]:
#cell 7
model = SentenceTransformer("all-mpnet-base-v2")

print("Transformer model loaded.")

Transformer model loaded.


In [8]:
#cell 8 
train_embed = model.encode(train["text"].tolist(), show_progress_bar=True)
val_embed = model.encode(val["text"].tolist(), show_progress_bar=True)
test_embed = model.encode(test["text"].tolist(), show_progress_bar=True)

print("Embedding shape:",train_embed.shape)

Batches:   0%|          | 0/74 [00:00<?, ?it/s]

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Embedding shape: (2338, 768)


In [9]:
#cell 9

X_train = np.hstack([train_rules, train_embed])
X_val = np.hstack([val_rules, val_embed])
X_test = np.hstack([test_rules, test_embed])

print("Hybrid feature shape:",X_train.shape)

Hybrid feature shape: (2338, 776)


In [10]:
#cell 10

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [11]:
#cell 11

y_train = train["label"].values
y_val = val["label"].values
y_test = test["label"].values

In [12]:
#cell 12

os.makedirs("phase3_features", exist_ok=True)

np.save("phase3_features/X_train.npy",X_train)
np.save("phase3_features/X_val.npy",X_val)
np.save("phase3_features/X_test.npy",X_test)

np.save("phase3_features/y_train.npy",y_train)
np.save("phase3_features/y_val.npy",y_val)
np.save("phase3_features/y_test.npy",y_test)

print("Phase 3 features saved.")

Phase 3 features saved.


In [13]:
print("Rule feature size:", train_rules.shape[1])

Rule feature size: 8
